In [1]:
from VerifyObjects import *
from VerifyRelation import *
from Verifier import *

In [2]:
from IPython.display import display, Math, Markdown

def printmd(veri_ob):
    display(Markdown("$\\displaystyle %s$" % repr(veri_ob)))

def display_prop():
    for p in Prop:
        display(p)

# Norm

In [3]:
class Norm(Scalar):
    def __init__(self, inside):
        self.inside = inside
        self.base = self
        self.is_nonnegative = 1

    def __new__(cls, inside):
        return Sqrt(NS(inside))

In [4]:
v = Vector('v')

nv = Norm(v)
display(nv)
display(type(nv))
display(NS(v)==nv**2)
nv.base.args[0]

||v||

VerifyObjects.ScalarPow

True

v

# Lemma 1, $\lambda_k$, $\theta_k$ positivity

In [5]:
def is_iterable(obj):
    try:
        iter(obj)
        return True
    except TypeError:
        return False

In [32]:
class Min(Scalar):
    def __init__(self, min_set):
        self.base = self
        self.min_set = min_set
        if is_iterable(min_set):
            minimality_ineqs = set()
            for element in min_set:
                minimality_ineqs.add( self <= element )
            self.facts["minimality"] = minimality_ineqs
        self.facts["is_contained_in"] = [self.min_set]

    def __str__(self):
        s = 'Min('
        if is_iterable(self.min_set):
            for element in self.min_set:
                s += str(element) + ', '
            s = s[:-2]
        s += ')'
        return s

    def __repr__(self):
        s = '\\text{Min} \left('
        if is_iterable(self.min_set):
            for element in self.min_set:
                s += repr(element) + ', \,\,'
            s = s[:-6]
        s += '\\right)'
        return s

In [33]:
def repeat_transivity_eq_le_lt(rel_list, prop_list):
    result_rel = rel_list[0]

    if not contained_in_props(result_rel, prop_list):
        raise Error(f'Should verify {result_rel} first')
        
    for i in range(len(rel_list)-1):
        if not contained_in_props(rel_list[i+1], prop_list):
            raise Error(f'Should verify {rel_list[i+1]} first')
        result_rel = transivity_eq_le_lt(result_rel, rel_list[i+1])
    return result_rel
## proof sequence가 참인 relation을 쓰는지 확인하는걸, Prop을 뒤지는거 말고 어떻게 하나?
## prop list를 추가적으로 넣는식으로 해야할듯

def contained_in_props(p, prop_list):
    # if p in Prop:
    #     return True
    for local_prop in prop_list:
        if p in local_prop:
            return True
    return False


def deduction(goal, proof, prop_list):
    if repeat_transivity_eq_le_lt(proof, prop_list)==goal:
        return True
    return False

In [34]:
infty = Scalar("∞")

def maximality_of_infinity(s): 
    if s.is_scalar==True:
        return s<=infty
    else:
        raise TypeError(f"{s} is not a scalar")
infty.facts["maximality"] = maximality_of_infinity

a = Scalar('a')
deduction(a<=infty, [infty.facts["maximality"](a)], [infty.facts])

Error: Should verify a <= ∞ first

In [35]:
# 0으로 나누려면, Fraction을 우리가 따로 짜야할듯...
# python에서는 0으로 나누면 터지잖아...
# extended real = True로 하면 0으로 나눈게 infinity가 되는 식이면 어떨까 싶음

In [36]:
x = Sequence('x')
lamda = ScalarSequence('\lambda')
theta = ScalarSequence('\\theta')
k = IterationCount('k')
f = Function('f')
g = Gradient(f)
x_star = x('\star')
f_star = f(x('\star'))

In [40]:
class VerifySet(set, VerifyObject):
    pass

min_set = VerifySet([ Sqrt(1+theta(k-1)) * lamda(k-1) , Norm(x(k) - x(k-1)) / ( 2 * Norm( g(x(k)) - g(x(k-1)) ) ) ])

min_set.facts['strict_lower_bound'] = set()
min_set.facts['props'] = set()
# 이름고민. 아니면 facts_dict, facts_set 이렇게 나눌까..


# 이건 target set이 iterable할때 문법. iterable 하지 않을때(finite이 아닐때)도 하는 방법을 고민해봐야할듯.
# 그러려면 forall e in S, e>lower_bound 에 해당 하는 문법이 뭘지 고민해봐야할듯.
def factsadd_strict_lower_bound(target_set, lower_bound):
    for e in target_set:
        if (e > lower_bound) not in Prop:
            if (e > lower_bound) not in target_set.facts['props']:
                raise Error(f"Should verify {e > lower_bound} first ")
    min_set.facts['strict_lower_bound'].add(lower_bound)

In [41]:
# Prop.add( Sqrt(1+theta(k-1)) * lamda(k-1) > 0 )
# Prop.add( Norm(x(k) - x(k-1)) / ( 2 * Norm( g(x(k)) - g(x(k-1)) ) ) > 0 )

min_set.facts['props'].add( Sqrt(1+theta(k-1)) * lamda(k-1) > 0 )
min_set.facts['props'].add( Norm(x(k) - x(k-1)) / ( 2 * Norm( g(x(k)) - g(x(k-1)) ) ) > 0 )

factsadd_strict_lower_bound(min_set, 0)

min_set.facts['strict_lower_bound']

{0}

In [42]:
def element_is_larger_than_strict_lower_bound(element, S, lower_bound):
    if S in element.facts["is_contained_in"]:
        if lower_bound in S.facts['strict_lower_bound']:
            # Prop.add(element > lower_bound)
            return element > lower_bound
        else:
            raise Error(f"{lower_bound} is not strict lower bound of {S}")
    else:
        raise Error(f"{element} is not contained in {S}")

element_is_larger_than_strict_lower_bound(Min(min_set), min_set, 0)                                        

0 < \text{Min} \left(\sqrt{ 1+\theta_{-1+k}}\lambda_{-1+k}, \,\,0.5||-x^{-1+k} + x^{k}||||-▽f(x^{-1+k}) + ▽f(x^{k})||^{-1.0}\right)

In [18]:
Min(min_set)

\text{Min} \left(\sqrt{ 1+\theta_{-1+k}}\lambda_{-1+k}, \,\,0.5||-x^{-1+k} + x^{k}||||-▽f(x^{-1+k}) + ▽f(x^{k})||^{-1.0}\right)

## Definition of materials

In [5]:
Prop.clear()